# 01 Clean data

Harmonize firm names and write cleaned tables to data/processed/.

Build data/processed/firm_crosswalk.csv mapping every raw manufacturer/applicant name
in PQR, WHO PQ FPP and WHO PQ API to a harmonized firm and corporate group.
Apply the firm crosswalk and write cleaned analysis tables to data/processed/.

In [ ]:
%run 00_utils.ipynb

import pandas as pd

## Firm crosswalk

In [ ]:
# one row per raw name per source
raw_names = pd.concat([
    load_pqr().groupby("manufacturer").size().rename("n_obs").reset_index()
        .rename(columns={"manufacturer": "raw_name"}).assign(source="pqr"),
    load_pq_fpp().groupby("Applicant").size().rename("n_obs").reset_index()
        .rename(columns={"Applicant": "raw_name"}).assign(source="pq_fpp"),
    load_pq_api().groupby("Applicant").size().rename("n_obs").reset_index()
        .rename(columns={"Applicant": "raw_name"}).assign(source="pq_api"),
], ignore_index=True)
raw_names["raw_name"] = raw_names["raw_name"].str.strip()
raw_names["key"] = raw_names["raw_name"].map(norm_key)

# apply the ordered firm rules to every raw name
rows = []
for row in raw_names.itertuples():
    matched = match(row.raw_name)
    if matched is None:
        rows.append((None, None, None, "junk"))
    else:
        rows.append(matched)
raw_names[["firm", "group", "hq_country", "method"]] = pd.DataFrame(rows, index=raw_names.index)

# fallback: names with no rule share a firm if their normalized keys match
# display name = the most frequent raw spelling within the key
unmatched = raw_names[raw_names.method == "normalized"]
display_names = unmatched.sort_values("n_obs", ascending=False).groupby("key")["raw_name"].first()
raw_names.loc[unmatched.index, "firm"] = unmatched["key"].map(display_names)
raw_names.loc[unmatched.index, "group"] = raw_names.loc[unmatched.index, "firm"]
raw_names.loc[unmatched.index, "method"] = unmatched["key"].map(unmatched.groupby("key").size()).map(
    lambda n_names: "normalized-merge" if n_names > 1 else "unchanged")

# keep track of how each headquarters country was assigned
raw_names["country_method"] = raw_names["hq_country"].notna().map({True: "researcher-coded", False: None})
missing_country = raw_names.hq_country.isna() & raw_names.raw_name.notna() & (raw_names.method != "junk")
raw_names.loc[missing_country, "hq_country"] = raw_names.loc[missing_country, "raw_name"].map(country_heuristic)
raw_names.loc[missing_country & raw_names.hq_country.notna(), "country_method"] = "name-heuristic"
raw_names = raw_names.drop(columns="key").sort_values(["firm", "source", "raw_name"])
output_path = PROCESSED / "firm_crosswalk.csv"
raw_names.to_csv(output_path, index=False)

print("raw names:", len(raw_names), "| junk:", (raw_names.method == "junk").sum())
for source in ["pqr", "pq_fpp", "pq_api"]:
    source_rows = raw_names[raw_names.source == source]
    print(f"{source:7s} raw {source_rows.raw_name.nunique():4d} -> firms {source_rows.firm.nunique():4d} -> groups {source_rows.group.nunique():4d}")
print("all sources: firms", raw_names.firm.nunique(), "groups", raw_names.group.nunique())
print(raw_names.method.value_counts().to_string())
spellings = raw_names.groupby("firm")["raw_name"].nunique()
print("\nfirms with >1 raw spelling:", (spellings > 1).sum())
print(raw_names[raw_names.method == "normalized-merge"][["raw_name", "firm", "source"]].to_string(index=False))

print("\nhq_country coverage by source (share of raw names):")
print(raw_names[raw_names.method != "junk"].groupby("source")["hq_country"].apply(lambda x: x.notna().mean()).round(2).to_string())

I matched every raw name to a firm and a corporate group, so spelling variants and mergers did not count as separate suppliers. The 326 raw names became 227 firms across the three sources. In PQR, 124 raw names became 110 firms and 104 groups. One name was a note and not a company, so I treated it as junk.

## Cleaned tables

In [ ]:
crosswalk = pd.read_csv(PROCESSED / "firm_crosswalk.csv")

# add firm, group and headquarters country to a raw table
def attach(table, name_column, source):
    firm_lookup = crosswalk[crosswalk.source == source].set_index("raw_name")[["firm", "group", "hq_country"]]
    merged = table.merge(firm_lookup, left_on=table[name_column].str.strip(), right_index=True, how="left")
    return merged.drop(columns="key_0", errors="ignore")

pqr = attach(load_pqr(), "manufacturer", "pqr")
pqr = pqr[pqr.firm.notna()]                       # drops 3 rows with an info-note instead of a name
pqr.to_csv(PROCESSED / "pqr_medicines_clean.csv", index=False)
fpp = attach(load_pq_fpp(), "Applicant", "pq_fpp")
fpp.to_csv(PROCESSED / "pq_fpp_clean.csv", index=False)
api = attach(load_pq_api(), "Applicant", "pq_api")
api.to_csv(PROCESSED / "pq_api_clean.csv", index=False)

# check coverage and concentration after cleaning
print("PQR rows", len(pqr), "| firms", pqr.firm.nunique(), "| groups", pqr.group.nunique())
spend = pqr.groupby("firm").cost_usd.sum()
print("spend share with hq_country:", round(spend[pqr.groupby("firm").hq_country.first().notna()].sum() / spend.sum(), 3))
print("\nspend share by HQ country:")
print((pqr.groupby("hq_country", dropna=False).cost_usd.sum() / pqr.cost_usd.sum()).sort_values(ascending=False).head(8).round(3).to_string())
in_pq = spend.index.isin(list(set(fpp.firm)))
print(f"\nPQR firms also WHO PQ FPP applicants: {in_pq.sum()} of {len(spend)} firms, "
      f"{spend[in_pq].sum() / spend.sum():.1%} of spend")
print("\ntop firms by spend:")
print((spend / spend.sum()).sort_values(ascending=False).head(8).round(3).to_string())

The clean PQR table had 74,689 medicine transactions after I dropped the 3 rows with a note in the manufacturer field. The HQ country was filled in for firms covering 97.8% of spend, so missing countries barely affect the spend shares. I coded HQ country by hand or from name patterns, so it still needs spot checks.

Firms based in India held 82% of spend, and firms based in China held 4.6%. The 46 PQR firms that are also WHO PQ applicants for finished products covered 96.2% of spend.